# Multi-Model Evaluation System
Supports:
- Model 1: Gas & Env (environmental + gas)
- Model 2: All (environmental + gas + specter)
- Model 3: Specter (spectral features)

**Late fusion** = simple average of model predictions (no weights)

In [1]:
from google.colab import files
uploaded = files.upload()

Saving random_forest_model_all.pkl to random_forest_model_all (3).pkl
Saving random_forest_model_gas&env.pkl to random_forest_model_gas&env (3).pkl
Saving random_forest_model_specter.pkl to random_forest_model_specter (3).pkl


In [2]:
import os
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

In [3]:
class SimpleModel:
    def __init__(self, name, features, model_path=None):
        self.name = name
        self.features = features
        self.model_path = model_path
        self.scaler = None  # default: no scaler

        try:
            if model_path and os.path.exists(model_path):
                print(f"Loading model for {name} from {model_path}")
                self.model = joblib.load(model_path)
            else:
                raise FileNotFoundError("Model path not found")
        except Exception as e:
            print(f"Using dummy model for {name} ({e})")
            self.scaler = StandardScaler()
            X_dummy = np.random.rand(50, len(self.features))
            y_dummy = np.random.randint(0, 5, 50)
            X_dummy = self.scaler.fit_transform(X_dummy)
            self.model = RandomForestClassifier(random_state=42)
            self.model.fit(X_dummy, y_dummy)

    def predict(self, values):
        X = np.array(values).reshape(1, -1)

        # Only apply scaling if scaler exists
        if self.scaler is not None:
            X = self.scaler.transform(X)

        try:
            pred = self.model.predict(X)
            return int(pred[0])
        except Exception as e:
            print(f"Error during prediction for {self.name}: {e}")
            return None


In [4]:
gas_env_features = ['Temp-int','Press-int','Humid-int','Temp-ext','Press-ext','Humid-ext','TGS20','TGS02','SGP']
specter_features = ['SpA410','SpB435','SpC460','SpD485','SpE510','SpF535','SpG560','SpH585','SpR610','SpI645','SpS680','SpJ705','SpT730','SpU760','SpV810','SpW860','SpK900','SpL940']
all_features = gas_env_features + specter_features

models = {
    "1": SimpleModel("Gas & Env", gas_env_features, "/content/random_forest_model_gas&env.pkl"),
    "2": SimpleModel("All", all_features, "/content/random_forest_model_all.pkl"),
    "3": SimpleModel("Specter", specter_features, "/content/random_forest_model_specter.pkl")
}


Loading model for Gas & Env from /content/random_forest_model_gas&env.pkl
Loading model for All from /content/random_forest_model_all.pkl
Loading model for Specter from /content/random_forest_model_specter.pkl


In [6]:
print("\nAvailable models:")
print("1 - Gas & Env")
print("2 - All")
print("3 - Specter")

chosen = input("\nEnter model numbers to evaluate (comma-separated, e.g., 1,2): ").split(",")
results = {}

for m in chosen:
    m = m.strip()
    if m not in models:
        print(f"Model {m} not found. Skipping.")
        continue

    model = models[m]
    num_features = len(model.features)
    print(f"\n➡ Enter input for {model.name}:")
    print(f"(Expected {num_features} comma-separated values)")

    # Example: 12.1,1013,45.3,...
    user_input_str = input("  Enter input values: ").strip()
    try:
        values = [float(x) for x in user_input_str.split(",")]
        if len(values) != num_features:
            print(f"Invalid input length! Expected {num_features} values, got {len(values)}.")
            continue
    except Exception as e:
        print(f"Could not parse input ({e})")
        continue

    pred = model.predict(values)
    results[model.name] = pred

if results:
    print("\nModel Predictions:")
    for name, pred in results.items():
        print(f"  {name}: Class {pred}")

    final = int(round(np.mean(list(results.values()))))
    print(f"\n Final (Late Fusion) Result: Class {final}")
else:
    print("\nNo valid models or inputs provided.")



Available models:
1 - Gas & Env
2 - All
3 - Specter

Enter model numbers to evaluate (comma-separated, e.g., 1,2): 1,2,3

➡ Enter input for Gas & Env:
(Expected 9 comma-separated values)
  Enter input values: 12.383920287730112,972.5930963740491,54.21227246036818,11.876704345276133,973.1922724603681,57.26528258957101,561.1960143865056,2653.75282589571,112.0

➡ Enter input for All:
(Expected 27 comma-separated values)
  Enter input values: 2.383920287730112,972.5930963740491,54.21227246036818,11.876704345276133,973.1922724603681,57.26528258957101,561.1960143865056,2653.75282589571,112.0,2.27226185798645,1.0007752180099487,0.7902707457542419,0.8125736713409424,4.217780113220215,7.5976080894470215,0.4497818052768707,0.42616045475006104,5.877471754111438e-39,0.3572191596031189,5.877471754111438e-39,0.35291603207588196,5.877471754111438e-39,5.877471754111438e-39,5.877471754111438e-39,0.7185888290405273,2.6577043533325195,0.844294548034668

➡ Enter input for Specter:
(Expected 18 comma-sepa